In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load the dataset
file_path = "C:\\Users\\ASUS\\Downloads\\amz_uk_processed_data.csv\\amz_uk_processed_data.csv"
df = pd.read_csv(file_path)

# 1. Remove duplicates
df.drop_duplicates(inplace=True)
print("Step 1/17: Duplicates removed")

# 2. Handle missing values
# Replace 0 with NaN in price and stars
df['price'] = df['price'].replace(0, np.nan)
df['stars'] = df['stars'].replace(0, np.nan)

# Fill numerical NaNs with median, boolean with False
df['price'] = df['price'].fillna(df['price'].median())
df['stars'] = df['stars'].fillna(0)
df['isBestSeller'] = df['isBestSeller'].fillna(False)
print("Step 2/17: Missing values handled")

# 3. Data type conversion
df = df.astype({
    'price': 'float32',
    'stars': 'float32',
    'reviews': 'int32',
    'boughtInLastMonth': 'int32'
})
print("Step 3/17: Data types converted")

# 4. Strip whitespace and lowercase
df['title'] = df['title'].str.strip().str.lower()
df['categoryName'] = df['categoryName'].str.strip().str.lower()
print("Step 4/17: Whitespace stripped and lowercased")

# 5. Remove special characters
df['title'] = df['title'].str.replace(r'[^a-z0-9\s]', '', regex=True)
print("Step 5/17: Special characters removed")

# 6. Outlier removal (99th percentile)
for col in ['price', 'reviews', 'boughtInLastMonth']:
    q = df[col].quantile(0.99)
    df = df[df[col] <= q]
print("Step 6/17: Outliers removed")

# 7. Log transformation
for col in ['price', 'reviews', 'boughtInLastMonth']:
    df[col] = np.log1p(df[col])
print("Step 7/17: Log transformation applied")

# 8. Normalize/scale values
scaler = StandardScaler()
df[['price', 'reviews']] = scaler.fit_transform(df[['price', 'reviews']])
print("Step 8/17: Values normalized")

# 9. Encode boolean
df['isBestSeller'] = df['isBestSeller'].astype(int)
print("Step 9/17: Boolean encoded")

# 10. Encode category
le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['categoryName'])
print("Step 10/17: Categories encoded")

# 11. Add popular feature
df['popular'] = ((df['stars'] > 4.0) & (df['reviews'] > 1000)).astype(int)
print("Step 11/17: Popular feature added")

# 12. Bucket star ratings
bins = [-1, 0, 2, 3, 4, 5]
labels = ['No rating', 'Low', 'Medium', 'Good', 'Excellent']
df['star_bucket'] = pd.cut(df['stars'], bins=bins, labels=labels)
print("Step 12/17: Star ratings bucketed")

# 13. Bucket price
price_bins = [0, 10, 30, float('inf')]
price_labels = ['Low', 'Medium', 'High']
df['price_bucket'] = pd.cut(df['price'], bins=price_bins, labels=price_labels)
print("Step 13/17: Price bucketed")

# 14. Clean URLs (placeholder - actual implementation requires validation)
df = df[df['imgUrl'].str.startswith('http', na=False)]
df = df[df['productURL'].str.startswith('http', na=False)]
print("Step 14/17: URLs cleaned")

# 15. Title Vectorization (TF-IDF placeholder - actual transformation during modeling)
# Skipped for now - will be done during modeling phase
print("Step 15/17: Title vectorization prepared (to be done during modeling)")

# 16. Drop unnecessary columns
df.drop(columns=['imgUrl', 'productURL', 'asin'], inplace=True)
print("Step 16/17: Columns dropped")

# 17. Text preprocessing for title
df['title'] = df['title'].str.replace(r'\s+', ' ', regex=True)  # Remove extra spaces
print("Step 17/17: Text preprocessing completed")

# Reset index and save memory
df.reset_index(drop=True, inplace=True)
df = df.astype({
    'category_encoded': 'int16',
    'popular': 'int8',
    'isBestSeller': 'int8'
})

# Save processed data
output_path = "C:\\Users\\ASUS\\Downloads\\amz_uk_processed_data_clean.csv"
df.to_csv(output_path, index=False)
print(f"Preprocessing complete! Cleaned data saved to {output_path}")
print(f"Final dataset shape: {df.shape}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Paths (edit if needed) ---
INPUT_PATH = r"C:\Users\ASUS\Downloads\amz_uk_processed_data_clean.csv"
OUTPUT_PATH = r"C:\Users\ASUS\Downloads\amz_uk_processed_data_with_vectors.csv"

# 1) Load
df = pd.read_csv(INPUT_PATH)

# 2) Safety checks + clean titles for vectorizer
if 'title' not in df.columns:
    raise KeyError("Column 'title' not found in the dataset.")

# Fill NaNs and force to string
df['title'] = df['title'].fillna('').astype(str).str.strip()

# (Optional but helpful) collapse multiple spaces
df['title'] = df['title'].str.replace(r'\s+', ' ', regex=True)

# 3) TF-IDF vectorization
# Adjust max_features/ngrams if you want bigger/smaller vectors
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=300,      # change to 1000/2000 if you want richer vectors
    ngram_range=(1, 2),    # unigrams + bigrams
    lowercase=True
)

tfidf_matrix = vectorizer.fit_transform(df['title'])

# 4) Store each row’s vector as a Python list (CSV will save it as a string)
# Note: this materializes the sparse matrix; fine for typical sizes.
X = tfidf_matrix.toarray()
df['title_vector'] = [X[i].tolist() for i in range(X.shape[0])]

# (Optional) keep the vector dimension for reference/debugging
df['title_vector_dim'] = X.shape[1]

# 5) Save
df.to_csv(OUTPUT_PATH, index=False)
print(f"Success! Added 'title_vector' ({X.shape[1]} dims) and saved to:\n{OUTPUT_PATH}")

# Quick sanity check
print("First vector (first 10 values):", df['title_vector'].iloc[0][:10] if len(df) else "No rows")

In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

# ================================
# 1. Load Your Dataset
# ================================
df = pd.read_csv("amz_uk_processed_data_with_dense_vectors.csv")  # <-- change this to your file path

# ================================
# 2. Convert 'title_vector' from string to numeric array
# ================================
df['title_vector'] = df['title_vector'].apply(ast.literal_eval)
X = np.vstack(df['title_vector'].values)
print("✅ Feature matrix shape:", X.shape)

# ================================
# 3. Find the Best Number of Clusters (Elbow Method)
# ================================
inertia = []
silhouette_scores = []
K_range = range(2, 11)  # test cluster sizes from 2 to 10

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)  # sum of squared distances to nearest cluster center
    silhouette_scores.append(silhouette_score(X, kmeans.labels_))

# Plot the Elbow Curve
plt.figure(figsize=(8, 4))
plt.plot(K_range, inertia, 'o-', label='Inertia (Elbow)')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal k")
plt.legend()
plt.show()

# Print silhouette scores to guide selection
for k, s in zip(K_range, silhouette_scores):
    print(f"Silhouette Score for k={k}: {s:.4f}")

# ================================
# 4. Fit KMeans with Chosen k
# ================================
best_k = int(input("Enter the number of clusters (k) based on elbow & silhouette score: "))
kmeans = KMeans(n_clusters=best_k, random_state=42)
df['relative_cluster'] = kmeans.fit_predict(X)

# ================================
# 5. Save Updated Dataset
# ================================
df.to_csv("your_dataset_with_clusters.csv", index=False)
print("✅ Dataset saved with 'relative_cluster' column!")

# ================================
# 6. Example: Show Few Products per Cluster
# ================================
print("\nSample products per cluster:")
for c in range(best_k):
    sample_rows = df[df['relative_cluster'] == c].head(3)
    print(f"\nCluster {c}:")
    for t in sample_rows['title']:
        print(f"  - {t}")

In [ ]:
# benchmark_system.py
"""
Comprehensive Benchmarking System for Product Recommendations
Comparing K-means Clustering vs LLM (Groq API)
Author: Research Project
Date: October 2025
"""

import os
import ast
import traceback
import re
import time
import json
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False
    print("WARNING: Groq SDK not available. Install with: pip install groq")

# =====================================================================
# CONFIGURATION
# =====================================================================

CONFIG = {
    # File paths
    'VECTORS_PATH': r"C:\Users\ASUS\proj 5\your_dataset_with_clusters.csv",
    'OUTPUT_DIR': r"C:\Users\ASUS\proj 5\benchmark_results",
    
    # API Keys - Replace with your actual keys
    'GROQ_API_KEYS': [
        "gsk_key5_replace_with_actual",
        "gsk_key6_replace_with_actual",
        "gsk_key7_replace_with_actual",
        "gsk_key8_replace_with_actual",
        "gsk_key9_replace_with_actual",
        "gsk_key10_replace_with_actual"
    ],
    
    # Model configuration
    'MODEL_NAME': "llama-3.1-8b-instant",  # Change as needed
    
    # Benchmark parameters
    'NUM_TEST_PRODUCTS': 100,
    'TOP_K': 5,
    'CONTEXT_LIMIT': 50,
    
    # API settings
    'API_TIMEOUT': 30,
    'MAX_RETRIES': 3,
    'RETRY_DELAY': 2,
}

# =====================================================================
# API KEY MANAGER
# =====================================================================

class APIKeyManager:
    """Manages multiple API keys with automatic rotation on quota exhaustion"""
    
    def __init__(self, api_keys: List[str]):
        self.api_keys = [key for key in api_keys if key and not key.startswith("gsk_key")]
        self.current_index = 0
        self.failed_keys = set()
        
        if not self.api_keys:
            raise ValueError("No valid API keys provided. Please replace placeholder keys.")
        
        print(f"Initialized with {len(self.api_keys)} API keys")
    
    def get_current_key(self) -> Optional[str]:
        """Get the current active API key"""
        if len(self.failed_keys) >= len(self.api_keys):
            return None
        
        while self.current_index in self.failed_keys:
            self.current_index = (self.current_index + 1) % len(self.api_keys)
        
        return self.api_keys[self.current_index]
    
    def mark_key_failed(self):
        """Mark current key as failed and rotate to next"""
        print(f"API key {self.current_index + 1} exhausted. Rotating...")
        self.failed_keys.add(self.current_index)
        self.current_index = (self.current_index + 1) % len(self.api_keys)
    
    def has_available_keys(self) -> bool:
        """Check if any keys are still available"""
        return len(self.failed_keys) < len(self.api_keys)

# =====================================================================
# DATA LOADER
# =====================================================================

def load_and_preprocess_data(path: str) -> Optional[pd.DataFrame]:
    """Load and preprocess the dataset with vectors"""
    try:
        print(f"\n{'='*60}")
        print("LOADING DATASET")
        print(f"{'='*60}")
        
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found at: {path}")
        
        df = pd.read_csv(path)
        print(f"✓ Loaded CSV with {len(df)} rows")
        
        # Check required columns
        required_cols = ['title', 'title_vector', 'relative_cluster']
        missing = [col for col in required_cols if col not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
        
        # Convert vectors
        print("✓ Converting vector strings to arrays...")
        first = df['title_vector'].iloc[0]
        if isinstance(first, str):
            df['title_vector'] = df['title_vector'].apply(
                lambda x: np.array(ast.literal_eval(x)) if pd.notna(x) else None
            )
        elif isinstance(first, list):
            df['title_vector'] = df['title_vector'].apply(
                lambda x: np.array(x) if pd.notna(x) else None
            )
        
        # Remove rows with invalid vectors
        df = df.dropna(subset=['title_vector'])
        df = df.reset_index(drop=True)
        
        print(f"✓ Dataset ready with {len(df)} products")
        print(f"  Vector dimension: {len(df['title_vector'].iloc[0])}")
        print(f"  Clusters: {df['relative_cluster'].nunique()}")
        
        return df
    
    except Exception as e:
        print(f"✗ Error loading data: {e}")
        traceback.print_exc()
        return None

# =====================================================================
# GROUND TRUTH GENERATION
# =====================================================================

def generate_ground_truth(df: pd.DataFrame, product_idx: int, top_k: int = 20) -> List[int]:
    """
    Generate ground truth for a product based on:
    1. Same cluster membership (primary)
    2. High cosine similarity (secondary)
    """
    try:
        product_cluster = df.at[product_idx, 'relative_cluster']
        product_vector = df.at[product_idx, 'title_vector'].reshape(1, -1)
        
        # Get all products in same cluster (excluding self)
        same_cluster = df[
            (df['relative_cluster'] == product_cluster) & 
            (df.index != product_idx)
        ]
        
        if len(same_cluster) == 0:
            return []
        
        # Calculate similarities
        cluster_vectors = np.stack(same_cluster['title_vector'].values)
        similarities = cosine_similarity(product_vector, cluster_vectors)[0]
        
        # Get top-k most similar from same cluster
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        ground_truth_indices = same_cluster.iloc[top_indices].index.tolist()
        
        return ground_truth_indices
    
    except Exception as e:
        print(f"Error generating ground truth for index {product_idx}: {e}")
        return []

# =====================================================================
# CLUSTERING-BASED RECOMMENDATIONS
# =====================================================================

def get_cluster_recommendations(
    df: pd.DataFrame, 
    product_idx: int, 
    top_k: int = 5
) -> Tuple[List[int], float]:
    """Get recommendations using K-means clustering approach"""
    start_time = time.time()
    
    try:
        product_cluster = df.at[product_idx, 'relative_cluster']
        product_vector = df.at[product_idx, 'title_vector'].reshape(1, -1)
        
        # Get cluster members
        cluster_products = df[
            (df['relative_cluster'] == product_cluster) & 
            (df.index != product_idx)
        ]
        
        if len(cluster_products) == 0:
            return [], time.time() - start_time
        
        # Calculate similarities
        cluster_vectors = np.stack(cluster_products['title_vector'].values)
        similarities = cosine_similarity(product_vector, cluster_vectors)[0]
        
        # Get top-k
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        recommendations = cluster_products.iloc[top_indices].index.tolist()
        
        latency = time.time() - start_time
        return recommendations, latency
    
    except Exception as e:
        print(f"Error in cluster recommendations: {e}")
        return [], time.time() - start_time

# =====================================================================
# LLM-BASED RECOMMENDATIONS
# =====================================================================

class LLMRecommender:
    """Handles LLM-based recommendations with API key rotation"""
    
    def __init__(self, api_key_manager: APIKeyManager, model_name: str):
        self.api_key_manager = api_key_manager
        self.model_name = model_name
        self.client = None
    
    def _get_client(self) -> Optional[Groq]:
        """Get or create Groq client with current API key"""
        if not GROQ_AVAILABLE:
            return None
        
        api_key = self.api_key_manager.get_current_key()
        if not api_key:
            return None
        
        try:
            return Groq(api_key=api_key)
        except Exception as e:
            print(f"Error creating Groq client: {e}")
            return None
    
    def _call_llm(self, prompt: str, max_retries: int = 3) -> Optional[str]:
        """Call LLM with retry logic and key rotation"""
        for attempt in range(max_retries):
            try:
                client = self._get_client()
                if not client:
                    return None
                
                response = client.chat.completions.create(
                    messages=[{"role": "user", "content": prompt}],
                    model=self.model_name,
                    temperature=0.1,
                    max_tokens=500,
                    timeout=CONFIG['API_TIMEOUT']
                )
                
                return response.choices[0].message.content.strip()
            
            except Exception as e:
                error_str = str(e).lower()
                
                # Check for rate limit errors
                if 'rate_limit' in error_str or 'quota' in error_str or '429' in error_str:
                    print(f"Rate limit hit. Rotating API key...")
                    self.api_key_manager.mark_key_failed()
                    
                    if not self.api_key_manager.has_available_keys():
                        print("All API keys exhausted!")
                        return None
                    continue
                
                # Other errors
                if attempt < max_retries - 1:
                    time.sleep(CONFIG['RETRY_DELAY'])
                    continue
                else:
                    print(f"LLM call failed after {max_retries} attempts: {e}")
                    return None
        
        return None
    
    def _parse_llm_output(self, llm_text: str, top_k: int) -> List[str]:
        """Parse LLM output to extract product titles"""
        try:
            # Try parsing as Python list
            parsed = ast.literal_eval(llm_text)
            if isinstance(parsed, list):
                return [str(p).strip() for p in parsed][:top_k]
        except:
            pass
        
        # Try line-by-line parsing
        lines = [l.strip(' -"\'') for l in llm_text.splitlines() if l.strip()]
        if len(lines) >= top_k:
            return lines[:top_k]
        
        # Try comma-separated
        parts = [p.strip() for p in llm_text.split(',') if p.strip()]
        return parts[:top_k]
    
    def get_recommendations(
        self, 
        df: pd.DataFrame, 
        product_idx: int, 
        top_k: int = 5
    ) -> Tuple[List[int], float]:
        """Get LLM-based recommendations"""
        start_time = time.time()
        
        try:
            product_title = df.at[product_idx, 'title']
            
            # Build context (sample of products for LLM awareness)
            context_titles = df['title'].sample(
                min(CONFIG['CONTEXT_LIMIT'], len(df))
            ).tolist()
            
            prompt = f"""Based on the product: "{product_title}"

Recommend exactly {top_k} similar products from an e-commerce perspective.

Available products (sample):
{context_titles}

Return ONLY a Python list of {top_k} product titles:
["product1", "product2", "product3", "product4", "product5"]"""
            
            # Call LLM
            llm_response = self._call_llm(prompt)
            if not llm_response:
                return [], time.time() - start_time
            
            # Parse response
            recommended_titles = self._parse_llm_output(llm_response, top_k)
            
            # Match titles to indices
            recommendation_indices = []
            for title in recommended_titles:
                # Fuzzy matching
                matches = df[
                    df['title'].str.contains(
                        re.escape(title[:50]), 
                        case=False, 
                        na=False, 
                        regex=True
                    )
                ]
                
                if len(matches) > 0:
                    recommendation_indices.append(matches.index[0])
            
            latency = time.time() - start_time
            return recommendation_indices, latency
        
        except Exception as e:
            print(f"Error in LLM recommendations: {e}")
            traceback.print_exc()
            return [], time.time() - start_time

# =====================================================================
# METRICS CALCULATION
# =====================================================================

def calculate_precision_recall(
    recommended: List[int], 
    ground_truth: List[int], 
    k: int = 5
) -> Tuple[float, float]:
    """Calculate Precision@k and Recall@k"""
    if not recommended or not ground_truth:
        return 0.0, 0.0
    
    recommended_set = set(recommended[:k])
    ground_truth_set = set(ground_truth)
    
    true_positives = len(recommended_set & ground_truth_set)
    
    precision = true_positives / len(recommended_set) if recommended_set else 0.0
    recall = true_positives / len(ground_truth_set) if ground_truth_set else 0.0
    
    return precision, recall

def calculate_mrr(recommended: List[int], ground_truth: List[int]) -> float:
    """Calculate Mean Reciprocal Rank"""
    ground_truth_set = set(ground_truth)
    
    for rank, rec_id in enumerate(recommended, start=1):
        if rec_id in ground_truth_set:
            return 1.0 / rank
    
    return 0.0

def calculate_ndcg(
    recommended: List[int], 
    ground_truth: List[int], 
    k: int = 5
) -> float:
    """Calculate Normalized Discounted Cumulative Gain@k"""
    ground_truth_set = set(ground_truth)
    
    # DCG
    dcg = 0.0
    for i, rec_id in enumerate(recommended[:k], start=1):
        relevance = 1.0 if rec_id in ground_truth_set else 0.0
        dcg += relevance / np.log2(i + 1)
    
    # IDCG (ideal)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(k, len(ground_truth)) + 1))
    
    return dcg / idcg if idcg > 0 else 0.0

# =====================================================================
# BENCHMARK RUNNER
# =====================================================================

class BenchmarkRunner:
    """Main benchmark execution class"""
    
    def __init__(self, df: pd.DataFrame, config: Dict):
        self.df = df
        self.config = config
        self.results = []
        
        # Initialize API key manager
        self.api_key_manager = APIKeyManager(config['GROQ_API_KEYS'])
        
        # Initialize LLM recommender
        self.llm_recommender = LLMRecommender(
            self.api_key_manager, 
            config['MODEL_NAME']
        )
    
    def run_benchmark(self) -> pd.DataFrame:
        """Run complete benchmark on first N products"""
        print(f"\n{'='*60}")
        print(f"STARTING BENCHMARK: {self.config['MODEL_NAME']}")
        print(f"{'='*60}")
        
        num_products = min(self.config['NUM_TEST_PRODUCTS'], len(self.df))
        print(f"Testing on first {num_products} products...")
        
        for idx in range(num_products):
            if idx % 10 == 0:
                print(f"\nProgress: {idx}/{num_products} products processed...")
            
            try:
                self._benchmark_single_product(idx)
            except Exception as e:
                print(f"Error processing product {idx}: {e}")
                continue
            
            # Save intermediate results every 25 products
            if (idx + 1) % 25 == 0:
                self._save_intermediate_results()
        
        print(f"\n✓ Completed {len(self.results)} product benchmarks")
        return self._create_results_dataframe()
    
    def _benchmark_single_product(self, product_idx: int):
        """Benchmark single product with both methods"""
        product_title = self.df.at[product_idx, 'title']
        
        # Generate ground truth
        ground_truth = generate_ground_truth(
            self.df, 
            product_idx, 
            top_k=20
        )
        
        if not ground_truth:
            return
        
        # Clustering recommendations
        cluster_recs, cluster_latency = get_cluster_recommendations(
            self.df, 
            product_idx, 
            self.config['TOP_K']
        )
        
        cluster_precision, cluster_recall = calculate_precision_recall(
            cluster_recs, ground_truth, self.config['TOP_K']
        )
        cluster_mrr = calculate_mrr(cluster_recs, ground_truth)
        cluster_ndcg = calculate_ndcg(cluster_recs, ground_truth, self.config['TOP_K'])
        
        # LLM recommendations
        llm_recs, llm_latency = self.llm_recommender.get_recommendations(
            self.df, 
            product_idx, 
            self.config['TOP_K']
        )
        
        llm_precision, llm_recall = calculate_precision_recall(
            llm_recs, ground_truth, self.config['TOP_K']
        )
        llm_mrr = calculate_mrr(llm_recs, ground_truth)
        llm_ndcg = calculate_ndcg(llm_recs, ground_truth, self.config['TOP_K'])
        
        # Store results
        self.results.append({
            'product_idx': product_idx,
            'product_title': product_title[:50],
            'ground_truth_size': len(ground_truth),
            
            # Clustering metrics
            'cluster_precision': cluster_precision,
            'cluster_recall': cluster_recall,
            'cluster_mrr': cluster_mrr,
            'cluster_ndcg': cluster_ndcg,
            'cluster_latency_ms': cluster_latency * 1000,
            
            # LLM metrics
            'llm_precision': llm_precision,
            'llm_recall': llm_recall,
            'llm_mrr': llm_mrr,
            'llm_ndcg': llm_ndcg,
            'llm_latency_ms': llm_latency * 1000,
        })
    
    def _create_results_dataframe(self) -> pd.DataFrame:
        """Convert results to DataFrame"""
        return pd.DataFrame(self.results)
    
    def _save_intermediate_results(self):
        """Save intermediate results to prevent data loss"""
        try:
            output_dir = self.config['OUTPUT_DIR']
            os.makedirs(output_dir, exist_ok=True)
            
            df_results = self._create_results_dataframe()
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"intermediate_results_{timestamp}.csv"
            filepath = os.path.join(output_dir, filename)
            
            df_results.to_csv(filepath, index=False)
            print(f"  → Saved intermediate results: {filename}")
        except Exception as e:
            print(f"Warning: Could not save intermediate results: {e}")

# =====================================================================
# RESULTS ANALYZER
# =====================================================================

class ResultsAnalyzer:
    """Analyze and visualize benchmark results"""
    
    def __init__(self, results_df: pd.DataFrame, config: Dict):
        self.results_df = results_df
        self.config = config
        self.output_dir = config['OUTPUT_DIR']
        os.makedirs(self.output_dir, exist_ok=True)
    
    def generate_summary_stats(self) -> Dict:
        """Calculate aggregate statistics"""
        print(f"\n{'='*60}")
        print("SUMMARY STATISTICS")
        print(f"{'='*60}")
        
        stats = {
            'model_name': self.config['MODEL_NAME'],
            'num_products_tested': len(self.results_df),
            
            # Clustering averages
            'cluster_avg_precision': self.results_df['cluster_precision'].mean(),
            'cluster_avg_recall': self.results_df['cluster_recall'].mean(),
            'cluster_avg_mrr': self.results_df['cluster_mrr'].mean(),
            'cluster_avg_ndcg': self.results_df['cluster_ndcg'].mean(),
            'cluster_avg_latency_ms': self.results_df['cluster_latency_ms'].mean(),
            
            # LLM averages
            'llm_avg_precision': self.results_df['llm_precision'].mean(),
            'llm_avg_recall': self.results_df['llm_recall'].mean(),
            'llm_avg_mrr': self.results_df['llm_mrr'].mean(),
            'llm_avg_ndcg': self.results_df['llm_ndcg'].mean(),
            'llm_avg_latency_ms': self.results_df['llm_latency_ms'].mean(),
        }
        
        # Print summary
        print(f"\nModel: {stats['model_name']}")
        print(f"Products Tested: {stats['num_products_tested']}")
        print(f"\n{'Method':<20} {'Precision@5':<15} {'Recall@5':<15} {'MRR':<15} {'nDCG@5':<15} {'Latency(ms)':<15}")
        print("-" * 95)
        print(f"{'K-means Clustering':<20} {stats['cluster_avg_precision']:<15.4f} {stats['cluster_avg_recall']:<15.4f} {stats['cluster_avg_mrr']:<15.4f} {stats['cluster_avg_ndcg']:<15.4f} {stats['cluster_avg_latency_ms']:<15.2f}")
        print(f"{stats['model_name']:<20} {stats['llm_avg_precision']:<15.4f} {stats['llm_avg_recall']:<15.4f} {stats['llm_avg_mrr']:<15.4f} {stats['llm_avg_ndcg']:<15.4f} {stats['llm_avg_latency_ms']:<15.2f}")
        
        return stats
    
    def save_results(self, stats: Dict):
        """Save all results to files"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Save detailed results
        detailed_file = os.path.join(
            self.output_dir, 
            f"detailed_results_{self.config['MODEL_NAME']}_{timestamp}.csv"
        )
        self.results_df.to_csv(detailed_file, index=False)
        print(f"\n✓ Saved detailed results: {detailed_file}")
        
        # Save summary stats
        summary_file = os.path.join(
            self.output_dir, 
            f"summary_stats_{self.config['MODEL_NAME']}_{timestamp}.json"
        )
        with open(summary_file, 'w') as f:
            json.dump(stats, f, indent=4)
        print(f"✓ Saved summary statistics: {summary_file}")
    
    def plot_all_graphs(self):
        """Generate all visualization graphs"""
        print(f"\n{'='*60}")
        print("GENERATING VISUALIZATIONS")
        print(f"{'='*60}")
        
        sns.set_style("whitegrid")
        plt.rcParams['figure.figsize'] = (14, 10)
        
        fig = plt.figure(figsize=(16, 12))
        
        # 1. Grouped Bar Chart: Precision & Recall
        ax1 = plt.subplot(2, 3, 1)
        self._plot_precision_recall_comparison(ax1)
        
        # 2. Grouped Bar Chart: MRR & nDCG
        ax2 = plt.subplot(2, 3, 2)
        self._plot_mrr_ndcg_comparison(ax2)
        
        # 3. Box Plot: Latency Distribution
        ax3 = plt.subplot(2, 3, 3)
        self._plot_latency_distribution(ax3)
        
        # 4. Scatter Plot: Latency vs Precision Trade-off
        ax4 = plt.subplot(2, 3, 4)
        self._plot_latency_precision_tradeoff(ax4)
        
        # 5. Line Plot: Performance Across Products
        ax5 = plt.subplot(2, 3, 5)
        self._plot_performance_trend(ax5)
        
        # 6. Heatmap: Metric Comparison
        ax6 = plt.subplot(2, 3, 6)
        self._plot_metric_heatmap(ax6)
        
        plt.tight_layout()
        
        # Save
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"benchmark_visualizations_{self.config['MODEL_NAME']}_{timestamp}.png"
        filepath = os.path.join(self.output_dir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"\n✓ Saved visualizations: {filename}")
        
        plt.show()
    
    def _plot_precision_recall_comparison(self, ax):
        """Bar chart comparing Precision & Recall"""
        methods = ['K-means', self.config['MODEL_NAME']]
        precision = [
            self.results_df['cluster_precision'].mean(),
            self.results_df['llm_precision'].mean()
        ]
        recall = [
            self.results_df['cluster_recall'].mean(),
            self.results_df['llm_recall'].mean()
        ]
        
        x = np.arange(len(methods))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, precision, width, label='Precision@5', color='#3498db')
        bars2 = ax.bar(x + width/2, recall, width, label='Recall@5', color='#e74c3c')
        
        ax.set_ylabel('Score', fontsize=11)
        ax.set_title('Precision & Recall Comparison', fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(methods)
        ax.legend()
        ax.set_ylim(0, 1.0)
        
        # Add value labels
        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    def _plot_mrr_ndcg_comparison(self, ax):
        """Bar chart comparing MRR & nDCG"""
        methods = ['K-means', self.config['MODEL_NAME']]
        mrr = [
            self.results_df['cluster_mrr'].mean(),
            self.results_df['llm_mrr'].mean()
        ]
        ndcg = [
            self.results_df['cluster_ndcg'].mean(),
            self.results_df['llm_ndcg'].mean()
        ]
        
        x = np.arange(len(methods))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, mrr, width, label='MRR', color='#2ecc71')
        bars2 = ax.bar(x + width/2, ndcg, width, label='nDCG@5', color='#f39c12')
        
        ax.set_ylabel('Score', fontsize=11)
        ax.set_title('MRR & nDCG Comparison', fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(methods)
        ax.legend()
        ax.set_ylim(0, 1.0)
        
        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    def _plot_latency_distribution(self, ax):
        """Box plot of latency distribution"""
        data = [
            self.results_df['cluster_latency_ms'],
            self.results_df['llm_latency_ms']
        ]
        
        bp = ax.boxplot(data, labels=['K-means', self.config['MODEL_NAME']],
                       patch_artist=True)
        
        colors = ['#3498db', '#e74c3c']
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax.set_ylabel('Latency (ms)', fontsize=11)
        ax.set_title('Latency Distribution', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
    
    def _plot_latency_precision_tradeoff(self, ax):
        """Scatter plot: Latency vs Precision trade-off"""
        ax.scatter(
            self.results_df['cluster_latency_ms'],
            self.results_df['cluster_precision'],
            alpha=0.6, s=50, label='K-means', color='#3498db'
        )
        ax.scatter(
            self.results_df['llm_latency_ms'],
            self.results_df['llm_precision'],
            alpha=0.6, s=50, label=self.config['MODEL_NAME'], color='#e74c3c'
        )
        
        ax.set_xlabel('Latency (ms)', fontsize=11)
        ax.set_ylabel('Precision@5', fontsize=11)
        ax.set_title('Latency vs Precision Trade-off', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)
    
    def _plot_performance_trend(self, ax):
        """Line plot showing performance across products"""
        products = self.results_df['product_idx'].values[:50]  # First 50
        
        ax.plot(products, 
               self.results_df['cluster_precision'].values[:50],
               marker='o', linestyle='-', label='Cluster Precision', 
               color='#3498db', alpha=0.7, markersize=4)
        ax.plot(products, 
               self.results_df['llm_precision'].values[:50],
               marker='s', linestyle='-', label='LLM Precision', 
               color='#e74c3c', alpha=0.7, markersize=4)
        
        ax.set_xlabel('Product Index', fontsize=11)
        ax.set_ylabel('Precision@5', fontsize=11)
        ax.set_title('Precision Trend (First 50 Products)', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)
    
    def _plot_metric_heatmap(self, ax):
        """Heatmap of normalized metrics"""
        metrics = ['Precision', 'Recall', 'MRR', 'nDCG']
        methods = ['K-means', 'LLM']
        
        data = np.array([
            [
                self.results_df['cluster_precision'].mean(),
                self.results_df['cluster_recall'].mean(),
                self.results_df['cluster_mrr'].mean(),
                self.results_df['cluster_ndcg'].mean()
            ],
            [
                self.results_df['llm_precision'].mean(),
                self.results_df['llm_recall'].mean(),
                self.results_df['llm_mrr'].mean(),
                self.results_df['llm_ndcg'].mean()
            ]
        ])
        
        im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
        
        ax.set_xticks(np.arange(len(metrics)))
        ax.set_yticks(np.arange(len(methods)))
        ax.set_xticklabels(metrics)
        ax.set_yticklabels(methods)
        
        # Add text annotations
        for i in range(len(methods)):
            for j in range(len(metrics)):
                text = ax.text(j, i, f'{data[i, j]:.3f}',
                             ha="center", va="center", color="black", fontsize=10)
        
        ax.set_title('Metric Heatmap', fontsize=12, fontweight='bold')
        plt.colorbar(im, ax=ax)

# =====================================================================
# MAIN EXECUTION
# =====================================================================

def main():
    """Main execution function"""
    print(f"\n{'#'*60}")
    print(f"# LLM vs CLUSTERING BENCHMARK SYSTEM")
    print(f"# Model: {CONFIG['MODEL_NAME']}")
    print(f"# Products: {CONFIG['NUM_TEST_PRODUCTS']}")
    print(f"{'#'*60}")
    
    try:
        # 1. Load data
        df = load_and_preprocess_data(CONFIG['VECTORS_PATH'])
        if df is None:
            print("Failed to load dataset. Exiting.")
            return
        
        # 2. Run benchmark
        benchmark = BenchmarkRunner(df, CONFIG)
        results_df = benchmark.run_benchmark()
        
        if len(results_df) == 0:
            print("No results generated. Exiting.")
            return
        
        # 3. Analyze results
        analyzer = ResultsAnalyzer(results_df, CONFIG)
        stats = analyzer.generate_summary_stats()
        
        # 4. Save results
        analyzer.save_results(stats)
        
        # 5. Generate visualizations
        analyzer.plot_all_graphs()
        
        print(f"\n{'='*60}")
        print("BENCHMARK COMPLETED SUCCESSFULLY!")
        print(f"{'='*60}")
        print(f"Results saved in: {CONFIG['OUTPUT_DIR']}")
        
    except KeyboardInterrupt:
        print("\n\nBenchmark interrupted by user.")
    except Exception as e:
        print(f"\n\nCritical error: {e}")
        traceback.print_exc()

if __name__ == "__main__":
    main()